
# CHƯƠNG 4: PHÂN TÍCH KHÁM PHÁ DỮ LIỆU (EDA)
## Pima Indians Diabetes Dataset




# Load thư viện và data

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('default')
sns.set()

df = pd.read_csv('pima-indians-diabetes.csv')
df.head()


## 4.1.1. Phân bố của từng biến số (Histogram)

In [ ]:

df.hist(figsize=(15,10), bins=20)
plt.tight_layout()
plt.show()



### Nhận xét:

- **Pregnancies**: Phân bố lệch phải, tập trung chủ yếu từ 0–5 lần mang thai.
- **Glucose**: Gần phân phối chuẩn, tập trung 90–150 mg/dL, có một số giá trị cao bất thường.
- **BloodPressure**: Phân bố tương đối chuẩn quanh 70–80 mmHg.
- **SkinThickness & Insulin**: Lệch phải mạnh, tồn tại nhiều ngoại lệ.
- **BMI**: Tập trung trong khoảng 25–40 (thừa cân – béo phì).
- **Age**: Chủ yếu từ 20–40 tuổi.
- **Outcome**: Mất cân bằng lớp (≈65% không mắc, 35% mắc).

➡ Dữ liệu y tế thể hiện tính không đồng đều cao (heterogeneity).


## 4.1.2. Phát hiện ngoại lệ (Boxplot)

In [ ]:

plt.figure(figsize=(15,8))
sns.boxplot(data=df)
plt.xticks(rotation=45)
plt.show()



### Nhận xét:

- **Insulin** có nhiều outliers nhất (>800).
- **BMI** có ngoại lệ cao (>50).
- **Pregnancies** có một số trường hợp >12 lần.
- Các ngoại lệ có thể phản ánh tình trạng sinh lý đặc biệt hoặc nhiễu dữ liệu.

➡ Nghiên cứu giữ nguyên outliers để đảm bảo tính đại diện mẫu.


## 4.2.1. Tỷ lệ mắc bệnh đái tháo đường

In [ ]:

df['Outcome'].value_counts().plot(kind='bar')
plt.title('Tỷ lệ mắc bệnh')
plt.show()

df['Outcome'].value_counts(normalize=True)



### Nhận xét:

- Không mắc: ≈65%
- Mắc bệnh: ≈35%

Tỷ lệ này cao hơn dân số chung (8–10%), phản ánh đặc thù nhóm Pima Indians.


## 4.2.2. So sánh đặc điểm hai nhóm

In [ ]:

df.groupby('Outcome').mean()



### Nhận xét:

- **Glucose**: Khác biệt lớn nhất (~30 mg/dL).
- **BMI**: Nhóm mắc bệnh cao hơn ~4 kg/m².
- **Age**: Nhóm mắc bệnh lớn tuổi hơn.
- **Insulin & Pregnancies**: Cao hơn ở nhóm mắc bệnh.

➡ Glucose, Age và BMI là các yếu tố dự báo mạnh.


## 4.3. Phân tích theo nhóm tuổi

In [ ]:

bins = [20,30,40,50,100]
labels = ['21-30','31-40','41-50','>50']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)

pd.crosstab(df['AgeGroup'], df['Outcome'], normalize='index')



### Nhận xét:

- Tỷ lệ mắc bệnh tăng theo tuổi.
- Nhóm >40 tuổi có nguy cơ cao rõ rệt.
- Ngay cả nhóm trẻ cũng có tỷ lệ đáng kể (~20%).

➡ Tuổi là yếu tố nguy cơ quan trọng.


## 4.4. Phân tích theo BMI (WHO)

In [ ]:

def bmi_category(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

df['BMI_Group'] = df['BMI'].apply(bmi_category)

pd.crosstab(df['BMI_Group'], df['Outcome'], normalize='index')



### Nhận xét:

- Tỷ lệ mắc bệnh tăng dần theo mức BMI.
- Nhóm béo phì có tỷ lệ mắc cao nhất.
- Béo phì làm tăng kháng insulin và viêm mãn tính.

➡ BMI là yếu tố nguy cơ có ý nghĩa lâm sàng mạnh.


## 4.4.3. BMI và Glucose theo Outcome

In [ ]:

sns.scatterplot(data=df, x='BMI', y='Glucose', hue='Outcome')
plt.show()



### Nhận xét:

- Điểm mắc bệnh tập trung ở vùng BMI cao + Glucose cao.
- Hai yếu tố này kết hợp giúp phân loại tốt hơn so với từng biến riêng lẻ.


## 4.5. Phân tích Glucose theo chuẩn ADA

In [ ]:

def glucose_category(g):
    if g < 100:
        return 'Normal'
    elif g < 126:
        return 'Prediabetes'
    else:
        return 'Diabetes'

df['Glucose_Group'] = df['Glucose'].apply(glucose_category)

pd.crosstab(df['Glucose_Group'], df['Outcome'], normalize='index')



### Nhận xét:

- Nhóm ≥126 mg/dL có tỷ lệ mắc bệnh cao nhất.
- Glucose là biến phân biệt mạnh nhất giữa hai nhóm.


## 4.6. Phân tích yếu tố di truyền

In [ ]:

sns.histplot(df['DiabetesPedigreeFunction'], bins=30)
plt.show()

df.groupby('Outcome')['DiabetesPedigreeFunction'].mean()



### Nhận xét:

- Phân phối lệch phải mạnh.
- Nhóm mắc bệnh có DPF trung bình cao hơn.
- Yếu tố di truyền làm tăng nguy cơ đáng kể khi kết hợp với Glucose cao.
